# Dual-Paradigm Framework (Symbolic vs Neural) | Cognitive Architectures

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, Literal
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
# ---- Symbolic rule engine (NO LLM — pure Python logic) ----
def check_ctr_threshold(amount: float) -> str:
    """Currency Transaction Report: required for transactions >= $10,000."""
    if amount >= 10_000:
        return f"VIOLATION: ${amount:,.2f} meets/exceeds $10,000 CTR threshold. Filing required."
    return f"COMPLIANT: ${amount:,.2f} is below $10,000 CTR threshold."

def check_wash_sale(sell_date: int, buy_date: int, sell_loss: bool) -> str:
    """Wash sale rule: can't claim loss if same security bought within 30 days."""
    days_apart = abs(buy_date - sell_date)
    if sell_loss and days_apart <= 30:
        return f"VIOLATION: Wash sale — repurchase {days_apart} days from sale (<=30). Loss not deductible."
    return f"COMPLIANT: {days_apart} days between transactions. No wash sale."

def check_structuring(amounts: list) -> str:
    """Anti-structuring: flag if multiple transactions just below $10K in same day."""
    suspicious = [a for a in amounts if 8_000 <= a < 10_000]
    if len(suspicious) >= 2:
        total = sum(suspicious)
        return f"ALERT: {len(suspicious)} transactions just below $10K (total: ${total:,.2f}). Possible structuring."
    return f"COMPLIANT: No structuring pattern detected."

RULES = {"ctr": check_ctr_threshold, "wash_sale": check_wash_sale, "structuring": check_structuring}

class DualState(TypedDict):
    query: str
    paradigm: NotRequired[str]
    response: NotRequired[str]

# Use pattern-based routing to avoid circular LLM dependency
SYMBOLIC_KEYWORDS = ["threshold", "rule", "compliance", "calculate", "limit",
                     "violation", "wash sale", "structuring", "ctr", "10,000", "10000"]

def meta_controller(state: DualState) -> Command[Literal["system1", "system2"]]:
    """Route: keyword match -> System 2 (symbolic), everything else -> System 1 (neural).
    Uses fast heuristic routing instead of LLM to avoid circular dependency."""
    query_lower = state["query"].lower()
    if any(kw in query_lower for kw in SYMBOLIC_KEYWORDS):
        paradigm = "system2"
    else:
        paradigm = "system1"
    return Command(goto=paradigm, update={"paradigm": paradigm})

def system1_neural(state: DualState) -> dict:
    """System 1 (Fast/Neural): LLM handles open-ended policy questions."""
    response = model.invoke(
        f"You are a financial compliance advisor. Answer this policy question:\n\n{state['query']}"
    )
    return {"response": f"[System 1 - Neural/LLM]\n{response.content}"}

def system2_symbolic(state: DualState) -> dict:
    """System 2 (Slow/Symbolic): Deterministic Python rules — NO LLM involved."""
    import re
    query = state["query"].lower()
    # Pattern-match to the right rule (in production, use a more robust parser)
    if "10,000" in query or "10000" in query or "ctr" in query or "reporting threshold" in query:
        # Extract amount from query
        amounts = re.findall(r'\$?([\d,]+(?:\.\d+)?)', state["query"])
        amount = float(amounts[0].replace(',', '')) if amounts else 0
        result = check_ctr_threshold(amount)
    elif "wash sale" in query:
        result = check_wash_sale(sell_date=0, buy_date=15, sell_loss=True)  # parsed from query
    elif "structuring" in query:
        result = check_structuring([9500, 9800, 9200])  # parsed from query
    else:
        result = "No matching symbolic rule. Falling back to general guidance."
    return {"response": f"[System 2 - Symbolic/Rules]\n{result}"}

In [5]:
graph = StateGraph(DualState)
graph.add_node("controller", meta_controller, destinations=("system1", "system2"))
graph.add_node("system1", system1_neural)
graph.add_node("system2", system2_symbolic)
graph.add_edge(START, "controller")
graph.add_edge("system1", END)
graph.add_edge("system2", END)

dual = graph.compile()

In [6]:
# Plot the workflow
plot_mermaid(dual)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	controller(controller)
	system1(system1)
	system2(system2)
	__end__([<p>__end__</p>]):::last
	__start__ --> controller;
	controller -.-> system1;
	controller -.-> system2;
	system1 --> __end__;
	system2 --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [7]:
# Open-ended policy question -> System 1 (LLM)
r1 = dual.invoke({"query": "What's our company policy on insider trading and quiet periods?"})
print(r1["response"][:200])

[System 1 - Neural/LLM]
As a financial compliance advisor, it is crucial to establish a clear and comprehensive policy on insider trading and quiet periods to ensure compliance with legal standards an


In [8]:
# Precise rule check -> System 2 (deterministic Python, no LLM)
r2 = dual.invoke({"query": "Does a $12,500 wire transfer violate the $10,000 CTR reporting threshold?"})
print(f"\n{r2['response']}")


[System 2 - Symbolic/Rules]
VIOLATION: $12,500.00 meets/exceeds $10,000 CTR threshold. Filing required.


In [9]:
stream_invoke(dual, {"query": "Does a $12,500 wire transfer violate the $10,000 CTR reporting threshold?"})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'query': 'Does a $12,500 wire transfer violate the $10,000 CTR reporting threshold?',
 'paradigm': 'system2',
 'response': '[System 2 - Symbolic/Rules]\nVIOLATION: $12,500.00 meets/exceeds $10,000 CTR threshold. Filing required.'}